# Approach 2 — A6: Disentangled Cross-Crop Disease Learning

**Goal (not a promised result):** Raise **official unseen Top-1** from the published Baseline 2 reference **9.60%** toward **20–30%** while reporting *seen*, *unseen*, *overall* and harmonic Top-1 under the same **333-candidate** protocol. The primary scientific evidence is **raw, train-only proxy-unseen performance**. Calibration is a **separate inference comparison**.

This notebook adapts the project's `Approach_2_Matched_Selection.ipynb` and keeps the team's **DINOv2 ViT-B/14**, 224×224 ImageNet normalization, **seed 42**, **batch 16**, **20 epochs**, AdamW, CosineAnnealingLR and crop–disease log-probability summation.

**Controlled ablations** (not a replication of any published model):

| Recipe | Change | Hypothesis |
|---|---|---|
| `B0_Matched` | Original frozen-backbone two linear heads | Reference/control |
| `E1_Disentangled` | Separate 128D crop/disease projections + decorrelation | Reduce crop leakage into disease features |
| `E2_Add_Prototypes` | E1 + leave-host-out training-only disease prototypes | Give cross-crop supervision even without same-disease batch mates |
| `E3_Add_SupCon` | E2 + cross-crop supervised contrastive loss | Further align disease embeddings across crops |

**Evaluation safety:** training, recipe/epoch selection and γ selection use **training CSV only**. A fresh final refit uses all 8,000 train images. Official test candidates are introduced **only after model locks**. Only filenames and `(crop,disease)` identities are accessed before prediction; per-image ground truth is accessed **after all predictions**. `RUN_FINAL_EVALUATION` defaults to `False`.

Research motivation (not claims of reproduced results): [Disentangling Before Composing (TPAMI 2025)](https://pubmed.ncbi.nlm.nih.gov/39466858/), [Graph-guided Cross-composition Disentanglement (ACL 2025)](https://aclanthology.org/2025.findings-acl.137/), [Prior Correction for GZSL (WACV 2025)](https://openaccess.thecvf.com/content/WACV2025/html/Bhat_PC-GZSL_Prior_Correction_for_Generalized_Zero_Shot_Learning_WACV_2025_paper.html).

> **Important:** Because earlier official test results have already been viewed during this project, further model development must use training-derived validation only. Ask the lecturer about a genuinely untouched final test or clearly disclose prior test exposure; do not iterate on official metrics.

## 1 · Run settings

Place this notebook in the team repository (any subfolder). Use its existing Python/Jupyter environment. Start with `DEBUG=True` and `RUN_FINAL_EVALUATION=False`, then run `DEBUG=False` for an actual development + refit. Only enable the **separate final evaluation** after you have committed to the locked recipe and weights. A new run stores all outputs under a unique timestamp.

In [ ]:
DEBUG = True                         # Smoke test first; False = full train+refit
RUN_FINAL_EVALUATION = False         # Keep False until teacher-approved final testing

from datetime import datetime, timezone
from pathlib import Path
import copy, hashlib, json, sys
import numpy as np
import pandas as pd
import timm
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from IPython.display import display
from PIL import Image

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'common' / 'dataset.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from inside COS30082-Plant-Disease-OOD repository.')
sys.path.insert(0, str(ROOT))
from common import config as team
from common.dataset import PlantDataset, get_loaders, get_pairs, get_transforms, load_split
from common.evaluate import PERTURBATIONS, top_k_accuracy, harmonic_mean, save_confusion

BACKBONE = 'vit_base_patch14_dinov2.lvd142m'
SEED = team.SEED
assert SEED == 42, 'This experiment assumes team Baseline 2 seed 42.'
DEVICE = team.DEVICE
team.NUM_WORKERS = 0
BATCH_SIZE = 8 if DEBUG else 16
EPOCHS = 1 if DEBUG else 20
LR, WEIGHT_DECAY, SMOOTHING = 1e-4, 1e-4, 0.1
EMBED_DIM = 128
DECOR_W = 0.02
PROTOTYPE_W = 0.15
SUPCON_W = 0.10
TEMP, PROTO_TEMP = 0.10, 0.10
GAMMA_GRID = [0.5 * i for i in range(25)]  # only for train-derived proxy tuning
RECIPES = {
    'B0_Matched': (False, False, False),
    'E1_Disentangled': (True, False, False),
    'E2_Add_Prototypes': (True, True, False),
    'E3_Add_SupCon': (True, True, True),
}
PROPOSALS = tuple(x for x in RECIPES if x != 'B0_Matched')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN_DIR = team.RESULTS_DIR / 'approach2_a6_disentangled_proto' / ('debug_' if DEBUG else '') / RUN_ID
DEV_DIR = RUN_DIR / 'development'
FINAL_DIR = RUN_DIR / 'final'
OFFICIAL_DIR = RUN_DIR / 'official'
CKPT_DIR = team.CHECKPOINT_DIR / 'approach2_a6_disentangled_proto' / RUN_ID
LOCK_PATH = FINAL_DIR / 'selection_lock.json'
BACKBONE_PATH = CKPT_DIR / 'frozen_backbone.pt'
assert (team.NUM_SEEN_PAIRS, team.NUM_UNSEEN_PAIRS, team.NUM_PAIRS) == (323,10,333)
if not team.TRAIN_CSV.exists() or not team.IMAGE_DIR.is_dir():
    raise FileNotFoundError(f'Dataset not found at {team.DATA_DIR}')
print('Device:', DEVICE, '| Batch:', BATCH_SIZE, '| Epochs:', EPOCHS, '| Run:', RUN_ID)

## 2 · Train-only splits and immutable metadata

Ten training-derived candidate compositions are excluded from the **development shadow models**. Five are used for epoch/recipe/γ tuning; the other five for **independent check**. Real official unseen pair identities are **not** read in this stage. The baseline model uses the same within-pair training/validation split as Baseline 2.

In [ ]:
def image_key(v):
    p = Path(str(v).strip().replace('\\', '/'))
    if not p.name or p.is_absolute() or '..' in p.parts or ':' in str(p):
        raise ValueError(f'Invalid relative image filename: {v!r}')
    return p.as_posix()

def sha256_file(p):
    with Path(p).open('rb') as f:
        return hashlib.file_digest(f, 'sha256').hexdigest()

def json_atomic(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp')
    tmp.write_text(json.dumps(obj, indent=2), encoding='utf-8')
    tmp.replace(path)

def torch_atomic(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp')
    torch.save(obj, tmp)
    tmp.replace(path)

train_full = load_split(team.TRAIN_CSV)
train_full['image'] = train_full.image.map(image_key)
if not train_full.image.is_unique:
    raise ValueError('Training filenames must be unique.')
seen_pairs = get_pairs()
assert seen_pairs.shape == (323,2)
_, eval_transform = get_transforms('imagenet')

def proxy_split(fit_frame, val_frame):
    counts = fit_frame.groupby('pair_row', sort=True).agg(
        n=('image','size'), crop=('crop','first'), disease=('disease','first'))
    crop_coverage = counts.crop.value_counts().to_dict()
    disease_coverage = counts.disease.value_counts().to_dict()
    shuffled = counts[counts.n >= 5].sample(frac=1, random_state=SEED+1)
    held = []
    for row_id, row in shuffled.iterrows():
        if crop_coverage[row.crop] > 1 and disease_coverage[row.disease] > 1:
            held.append(int(row_id))
            crop_coverage[row.crop] -= 1
            disease_coverage[row.disease] -= 1
        if len(held) == 10:
            break
    if len(held) != 10:
        raise RuntimeError('Cannot build 10 valid train-only proxy combinations.')
    others = val_frame.loc[~val_frame.pair_row.isin(held)].sample(frac=1, random_state=SEED+2)
    half = len(others)//2
    groups = {
        'tune_seen': others.iloc[:half],
        'check_seen': others.iloc[half:],
        'tune_proxy': train_full.loc[train_full.pair_row.isin(held[:5])],
        'check_proxy': train_full.loc[train_full.pair_row.isin(held[5:])],
    }
    if any(f.empty for f in groups.values()):
        raise RuntimeError('Empty proxy development subgroup.')
    fit_mask = torch.ones(len(seen_pairs),dtype=torch.bool)
    fit_mask[held] = False
    return fit_mask, groups

SETTINGS = {
    'backbone':BACKBONE, 'seed':SEED, 'batch_size':BATCH_SIZE, 'epochs':EPOCHS,
    'learning_rate':LR, 'weight_decay':WEIGHT_DECAY, 'smoothing':SMOOTHING,
    'embedding_dim':EMBED_DIM, 'decor_weight':DECOR_W, 'prototype_weight':PROTOTYPE_W,
    'supcon_weight':SUPCON_W, 'contrast_temperature':TEMP, 'prototype_temperature':PROTO_TEMP,
    'gamma_grid':GAMMA_GRID, 'recipes':{k:list(v) for k,v in RECIPES.items()},
    'train_csv_sha256':sha256_file(team.TRAIN_CSV),'torch':str(torch.__version__),
    'timm':timm.__version__, 'debug':DEBUG,
    'selection':'baseline: seen val; proposed: train-only tune raw harmonic',
    'full_refit':'fresh pretrained frozen backbone; new heads; all train rows; selected epochs'
}

## 3 · Heads, disentanglement, cross-host prototypes and SupCon

The B0 control has the **same two linear heads** as Baseline 2. Proposed models project backbone features separately to 128 dimensions. Only the proposed models receive feature-decorrelation loss; E2+ align a disease embedding with **training-derived prototypes from other crops**; E3 additionally uses cross-crop SupCon when minibatches contain same-disease, different-crop anchors. No test metadata enters prototypes or losses.

In [ ]:
class Heads(nn.Module):
    def __init__(self, width, disentangled):
        super().__init__()
        self.disentangled = disentangled
        if disentangled:
            self.crop_proj = nn.Sequential(nn.LayerNorm(width),nn.Linear(width,EMBED_DIM),nn.GELU())
            self.disease_proj = nn.Sequential(nn.LayerNorm(width),nn.Linear(width,EMBED_DIM),nn.GELU())
            width = EMBED_DIM
        self.crop_head = nn.Linear(width, team.NUM_CROPS)
        self.disease_head = nn.Linear(width, team.NUM_DISEASES)

    def forward(self, features):
        if self.disentangled:
            c = self.crop_proj(features)
            d = self.disease_proj(features)
        else:
            c = d = features
        return self.crop_head(c), self.disease_head(d), c, d

def compose_scores(crop_logits, disease_logits, candidate_pairs):
    p = candidate_pairs.to(crop_logits.device)
    c_log = F.log_softmax(crop_logits.float(),dim=1)
    d_log = F.log_softmax(disease_logits.float(),dim=1)
    return c_log[:,p[:,0]] + d_log[:,p[:,1]]

def feature_decorrelation(crop_z, disease_z):
    def normalize(z):
        z = z - z.mean(0,keepdim=True)
        return z / (z.square().mean(0,keepdim=True) + 1e-4).sqrt()
    corr = normalize(crop_z).T @ normalize(disease_z) / len(crop_z)
    return corr.square().mean()

def cross_crop_supcon(z, crop, disease):
    z = F.normalize(z,dim=1)
    same_dis = disease[:,None].eq(disease[None,:])
    pos = same_dis & crop[:,None].ne(crop[None,:])
    neg = ~same_dis
    sim = (z @ z.T) / TEMP
    denom = sim.masked_fill(~(pos | neg), -1e9).logsumexp(1)
    count = pos.sum(1)
    mean_pos = sim.masked_fill(~pos,0).sum(1)/count.clamp_min(1)
    valid = count.gt(0) & neg.any(1)
    return (denom-mean_pos).masked_fill(~valid,0).sum()/valid.sum().clamp_min(1),int(valid.sum())

class CrossHostPrototypes:
    """One *frozen* backbone mean for each seen crop-disease training pair.
    When matching a query crop, exclude all prototypes from that crop.
    """
    def __init__(self, pair_means, pair_ids):
        self.pair_means = pair_means.detach()
        self.disease_ids = pair_ids[:,1]
        self.host_counts = torch.bincount(self.disease_ids,minlength=team.NUM_DISEASES)
        self.lookup = torch.zeros((team.NUM_CROPS,team.NUM_DISEASES),dtype=torch.long,
                                  device=pair_means.device)
        self.lookup[pair_ids[:,0],pair_ids[:,1]] = torch.arange(
            1,len(pair_ids)+1,device=pair_means.device)

    def loss(self, heads, features, crop_ids, disease_ids):
        lookup = self.lookup[crop_ids]                        # [B, D], 0 = missing
        allowed_count = self.host_counts[None,:]-lookup.gt(0).long()
        valid = allowed_count.gather(1,disease_ids[:,None]).squeeze(1).gt(0)
        valid &= allowed_count.gt(0).sum(1).gt(1)
        if not valid.any():
            return features.sum()*0, 0
        # Project pretrained *fit-only* prototypes through the same learned disease space.
        proto = F.normalize(heads.disease_proj(self.pair_means),dim=1)
        totals = proto.new_zeros((team.NUM_DISEASES,proto.shape[1]))
        totals.index_add_(0,self.disease_ids,proto)
        own = F.pad(proto,(0,0,1,0))[lookup[valid]]
        ext = (totals[None,:,:]-own)/allowed_count[valid].clamp_min(1)[:,:,None]
        ext = F.normalize(ext,dim=2)
        query = F.normalize(heads.disease_proj(features[valid]),dim=1)
        logits = (ext*query[:,None,:]).sum(-1)/PROTO_TEMP
        logits = logits.masked_fill(allowed_count[valid].eq(0),-1e6)
        return F.cross_entropy(logits,disease_ids[valid]), int(valid.sum())

@torch.inference_mode()
def build_prototypes(backbone, fit_frame):
    loader = DataLoader(PlantDataset(fit_frame,eval_transform,'imagenet'),
                        batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
    sums = torch.zeros((len(seen_pairs),backbone.num_features),device=DEVICE)
    counts = torch.zeros(len(seen_pairs),device=DEVICE)
    for images,_,_,pair_rows in loader:
        idx = pair_rows.to(DEVICE)
        feats = backbone(images.to(DEVICE)).float()
        sums.index_add_(0,idx,feats)
        counts.index_add_(0,idx,torch.ones_like(idx,dtype=counts.dtype))
    if int(counts.sum().item()) != len(fit_frame):
        raise RuntimeError('Prototype construction skipped training samples.')
    active = counts.gt(0)
    return CrossHostPrototypes(sums[active]/counts[active,None],seen_pairs.to(DEVICE)[active])

def snapshot(head):
    return {k:v.detach().cpu().clone() for k,v in head.state_dict().items()}

@torch.inference_mode()
def cache_group(backbone,frame):
    loader = DataLoader(PlantDataset(frame,eval_transform,'imagenet'),batch_size=BATCH_SIZE,
                        shuffle=False,num_workers=0)
    feats, labels = [],[]
    for images,_,_,target in loader:
        feats.append(backbone(images.to(DEVICE)).float().cpu())
        labels.append(target.cpu())
    return torch.cat(feats),torch.cat(labels)

@torch.inference_mode()
def eval_head(head,group):
    feats, targets = group
    crop,disease,_,_ = head(feats.to(DEVICE))
    scores = compose_scores(crop,disease,seen_pairs).cpu()
    return scores,targets,disease.argmax(1).cpu()

def train_one_step(head, opt, features, crop, disease, prototype, recipe):
    uses_projection,uses_proto,uses_supcon = RECIPES[recipe]
    c_log,d_log,c_z,d_z = head(features)
    loss = F.cross_entropy(c_log,crop,label_smoothing=SMOOTHING)
    loss += F.cross_entropy(d_log,disease,label_smoothing=SMOOTHING)
    proto_count=supcon_count=0
    if uses_projection:
        loss = loss+DECOR_W*feature_decorrelation(c_z,d_z)
    if uses_proto:
        value,proto_count = prototype.loss(head,features,crop,disease)
        loss = loss+PROTOTYPE_W*value
    if uses_supcon:
        value,supcon_count = cross_crop_supcon(d_z,crop,disease)
        loss = loss+SUPCON_W*value
    if not torch.isfinite(loss):
        raise FloatingPointError(f'Nonfinite loss in {recipe}.')
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    return loss.item(),proto_count,supcon_count

## 4 · Train-only development and model selection

Four recipes share the **same frozen DINOv2 forward** for each augmented minibatch. Proposed heads start from identical weights. The `B0_Matched` epoch is selected on seen validation (as in Baseline 2); proposed recipes are compared on the **raw 323-candidate seen/proxy-unseen harmonic mean**. Calibration γ is selected from **development proxy scores only** and then checked on a disjoint set. The independent check is *not* used for ranking hyperparameters.

This training section **never opens `test_seen.csv` or `test_unseen.csv`**.

In [ ]:
def metric_group(seen_group, proxy_group, fit_mask, gamma=0.0):
    s,sy,_ = seen_group
    u,uy,d_pred = proxy_group
    penalty = gamma*fit_mask.to(s.dtype)
    seen = top_k_accuracy(s-penalty,sy,1)
    unseen = top_k_accuracy(u-penalty,uy,1)
    return {'seen_top1':seen,'proxy_unseen_top1':unseen,
            'harmonic_top1':harmonic_mean(seen,unseen),
            'proxy_disease_head_top1':round(100*d_pred.eq(seen_pairs[uy,1]).float().mean().item(),2)}

def proposal_key(row):
    return (row['harmonic_top1'],row['proxy_unseen_top1'],row['seen_top1'],-row['epoch'])

def select_gamma(tune_groups,check_groups,fit_mask):
    grid=[]
    for gamma in GAMMA_GRID:
        grid.append({'gamma':gamma,**metric_group(*tune_groups,fit_mask,gamma)})
    chosen=max(grid,key=lambda r:(r['harmonic_top1'],r['proxy_unseen_top1'],
                                  r['seen_top1'],-r['gamma']))['gamma']
    checks=[{'scoring':name,'gamma':gamma,**metric_group(*check_groups,fit_mask,gamma)}
            for name,gamma in [('raw',0.0),('calibrated',chosen)]]
    return chosen,grid,checks

@torch.no_grad()
def create_backbone():
    model = timm.create_model(BACKBONE,pretrained=True,num_classes=0,
                              img_size=team.IMAGE_SIZE)
    model.requires_grad_(False).eval()
    return model.to(DEVICE)

def make_heads(width):
    team.set_seed()
    base = Heads(width,False).to(DEVICE)
    projected = Heads(width,True).to(DEVICE)
    heads = {'B0_Matched':base}
    for recipe in PROPOSALS:
        heads[recipe] = copy.deepcopy(projected)
    return heads

def train_recipes(backbone,loader,fit_mask=None, tune=None, epochs=None, val_loader=None):
    """Train all recipes on one shared sequence of image augmentations."""
    if epochs is None: epochs={name:EPOCHS for name in RECIPES}
    units=make_heads(backbone.num_features)
    options={name:torch.optim.AdamW(head.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
             for name,head in units.items()}
    schedules={name:torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS)
               for name,opt in options.items()}
    frame = loader.dataset.frame
    fit_frame = (frame.loc[frame.pair_row.isin(torch.where(fit_mask)[0].tolist())]
                 if fit_mask is not None else frame)
    # Prototype construction is fit-only. No sample from excluded pairs enters.
    proto = build_prototypes(backbone,fit_frame)
    fit_device = fit_mask.to(DEVICE) if fit_mask is not None else None
    history=[]
    best={}
    for epoch in range(1,max(epochs.values())+1):
        active = [name for name in RECIPES if epoch<=epochs[name]]
        totals={name:[0.,0,0,0] for name in active}
        seen_count=0
        for bi,(images,crop,disease,pair_row) in enumerate(loader,1):
            crop,disease,pair_row=crop.to(DEVICE),disease.to(DEVICE),pair_row.to(DEVICE)
            if fit_device is not None:
                keep=fit_device[pair_row]
                if not keep.any():continue
                images=images[keep.cpu()]
                crop,disease=crop[keep],disease[keep]
            with torch.no_grad():
                features=backbone(images.to(DEVICE)).float()
            seen_count+=len(images)
            for name in active:
                units[name].train()
                loss,pro_count,sup_count=train_one_step(units[name],options[name],features,crop,disease,proto,name)
                sums=totals[name]
                sums[0]+=loss*len(images)
                sums[1]+=len(images)
                sums[2]+=pro_count
                sums[3]+=sup_count
            if bi==1 or bi%75==0 or bi==len(loader):
                print(f'epoch {epoch}/{max(epochs.values())}, batch {bi}/{len(loader)}',flush=True)
        if seen_count!=len(fit_frame):
            raise RuntimeError(f'Expected {len(fit_frame)} fit images, got {seen_count}')
        for name in active:
            schedules[name].step()
            head=units[name].eval()
            used=totals[name]
            row={'recipe':name,'epoch':epoch,'train_loss':used[0]/used[1],
                 'prototype_anchor_fraction':used[2]/used[1],
                 'supcon_anchor_fraction':used[3]/used[1]}
            if tune is not None:
                summary=metric_group(eval_head(head,tune[0]),eval_head(head,tune[1]),fit_mask)
                row.update(summary)
                if name not in best or proposal_key(row)>proposal_key(best[name]['metrics']):
                    best[name]={'metrics':row.copy(),'weights':snapshot(head)}
            elif val_loader is not None and name=='B0_Matched':
                # Only needed for development Baseline 2 control.
                pass
            history.append(row)
        print(pd.DataFrame(history[-len(active):]).to_string(index=False),flush=True)
    return units,history,best

# We use one shared backbone and two head suites (main and proxy) with identical
# initializations and minibatch streams, following the uploaded matched-selection sample.
def development_run():
    team.set_seed()
    train_loader,val_loader,test_seen,test_unseen = get_loaders(
        BATCH_SIZE,norm='imagenet',debug=DEBUG)
    if test_seen is not None or test_unseen is not None:
        raise RuntimeError('Development must not access official test loaders.')
    train_loader.pin_memory=val_loader.pin_memory=False
    backbone=create_backbone()
    fit_mask,groups = (None,{}) if DEBUG else proxy_split(
        train_loader.dataset.frame,val_loader.dataset.frame)
    main_proto = build_prototypes(backbone,train_loader.dataset.frame)
    shadow_frame = (train_loader.dataset.frame.loc[
        train_loader.dataset.frame.pair_row.isin(torch.where(fit_mask)[0].tolist())]
        if fit_mask is not None else None)
    shadow_proto = build_prototypes(backbone,shadow_frame) if shadow_frame is not None else None
    main,shadow=make_heads(backbone.num_features), ({} if DEBUG else make_heads(backbone.num_features))
    main_opt={k:torch.optim.AdamW(v.parameters(),lr=LR,weight_decay=WEIGHT_DECAY) for k,v in main.items()}
    sh_opt={k:torch.optim.AdamW(v.parameters(),lr=LR,weight_decay=WEIGHT_DECAY) for k,v in shadow.items()}
    main_sch={k:torch.optim.lr_scheduler.CosineAnnealingLR(v,T_max=EPOCHS) for k,v in main_opt.items()}
    sh_sch={k:torch.optim.lr_scheduler.CosineAnnealingLR(v,T_max=EPOCHS) for k,v in sh_opt.items()}
    tune_cache=({key:cache_group(backbone,groups[key]) for key in ('tune_seen','tune_proxy')}
                if not DEBUG else {})
    check_cache=({key:cache_group(backbone,groups[key]) for key in ('check_seen','check_proxy')}
                if not DEBUG else {})
    baseline_val=cache_group(backbone,val_loader.dataset.frame)
    history=[]; best={}
    fit_device=fit_mask.to(DEVICE) if fit_mask is not None else None
    for epoch in range(1,EPOCHS+1):
        sums={name:[0.,0,0] for name in RECIPES}
        for bi,(images,crop,disease,rows) in enumerate(train_loader,1):
            with torch.no_grad():features=backbone(images.to(DEVICE)).float()
            crop,disease,rows=(t.to(DEVICE) for t in (crop,disease,rows))
            keep=fit_device[rows] if fit_device is not None else None
            for name in RECIPES:
                loss,pa,sa=train_one_step(main[name],main_opt[name],features,crop,disease,main_proto,name)
                sums[name][0]+=loss*len(images);sums[name][1]+=pa;sums[name][2]+=sa
                if keep is not None and keep.any():
                    train_one_step(shadow[name],sh_opt[name],features[keep],crop[keep],
                                   disease[keep],shadow_proto,name)
            if bi==1 or bi%75==0 or bi==len(train_loader):
                print(f'epoch {epoch}/{EPOCHS} | batch {bi}/{len(train_loader)}',flush=True)
        for s in (*main_sch.values(),*sh_sch.values()):s.step()
        for name in RECIPES:
            main[name].eval()
            seen_val = eval_head(main[name],baseline_val)
            val_top1=top_k_accuracy(seen_val[0],seen_val[1],1)
            row={'recipe':name,'epoch':epoch,'train_loss':sums[name][0]/len(train_loader.dataset),
                 'supcon_anchor_fraction':sums[name][2]/len(train_loader.dataset),
                 'prototype_anchor_fraction':sums[name][1]/len(train_loader.dataset),
                 'baseline_split_seen_top1':val_top1}
            if not DEBUG:
                shadow[name].eval()
                train_only = metric_group(eval_head(shadow[name],tune_cache['tune_seen']),
                                          eval_head(shadow[name],tune_cache['tune_proxy']),fit_mask)
                row.update(train_only)
            key=(val_top1,-epoch) if name=='B0_Matched' or DEBUG else proposal_key(row)
            if name not in best or key>best[name]['key']:
                best[name]={'metrics':row.copy(),'weights':snapshot(main[name]),
                            'shadow':snapshot(shadow[name]) if not DEBUG else None, 'key':key}
            history.append(row)
        print(pd.DataFrame(history[-len(RECIPES):]).to_string(index=False),flush=True)
    DEV_DIR.mkdir(parents=True,exist_ok=True)
    pd.DataFrame(history).to_csv(DEV_DIR/'training_history.csv',index=False)
    if DEBUG:
        print('DEBUG completed: no model selection, full refit, or official testing.')
        return None
    chosen=max(PROPOSALS,key=lambda r:proposal_key(best[r]['metrics']))
    print('Selected proposed recipe:',chosen,'epoch:',best[chosen]['metrics']['epoch'])
    penalties={};check_rows=[];grid_rows=[]
    for recipe in ('B0_Matched',chosen):
        shadow[recipe].load_state_dict(best[recipe]['shadow'])
        tuned=(eval_head(shadow[recipe],tune_cache['tune_seen']),
               eval_head(shadow[recipe],tune_cache['tune_proxy']))
        checked=(eval_head(shadow[recipe],check_cache['check_seen']),
                 eval_head(shadow[recipe],check_cache['check_proxy']))
        gamma,grid,checks=select_gamma(tuned,checked,fit_mask)
        penalties[recipe]=gamma
        grid_rows.extend({'recipe':recipe,**r} for r in grid)
        check_rows.extend({'recipe':recipe,**r} for r in checks)
    pd.DataFrame(grid_rows).to_csv(DEV_DIR/'calibration_grid.csv',index=False)
    pd.DataFrame(check_rows).to_csv(DEV_DIR/'independent_proxy_check.csv',index=False)
    result={'settings':SETTINGS,'primary_recipe':chosen,
            'selected':{k:v['metrics'] for k,v in best.items()},
            'gamma':penalties,
            'tune_proxy_rows':sorted(groups['tune_proxy'].pair_row.unique().tolist()),
            'check_proxy_rows':sorted(groups['check_proxy'].pair_row.unique().tolist()),
            'seen_pairs':seen_pairs.tolist()}
    json_atomic(DEV_DIR/'development_selection.json',result)
    return result

# Runs training & development ONLY; never loads official unseen or seen test data.
development = development_run()

## 5 · Fresh full-training refit and locks

After development, train the B0 control and the **chosen proposed recipe** again from fresh pretrained-backbone/head initialisation on **all 8,000 train images**, using their previously selected epoch counts. Save the frozen backbone once and each recipe's head separately. No official CSV is accessed here.

In [ ]:
def full_training_refit(development):
    if DEBUG or development is None:
        print('No full refit in DEBUG mode.')
        return None
    if LOCK_PATH.exists():
        raise FileExistsError('This run has already been locked. Do not overwrite it.')
    if sha256_file(team.TRAIN_CSV)!=SETTINGS['train_csv_sha256']:
        raise RuntimeError('Training dataset changed since development.')
    selected_recipes=('B0_Matched',development['primary_recipe'])
    epochs={r:int(development['selected'][r]['epoch']) for r in selected_recipes}
    team.set_seed()
    backbone=create_backbone()
    heads=make_heads(backbone.num_features)
    heads={r:heads[r] for r in selected_recipes}
    optimizers={r:torch.optim.AdamW(h.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
                for r,h in heads.items()}
    schedulers={r:torch.optim.lr_scheduler.CosineAnnealingLR(o,T_max=EPOCHS)
                for r,o in optimizers.items()}
    prototype=build_prototypes(backbone,train_full)
    train_transform,_=get_transforms('imagenet')
    loader=DataLoader(PlantDataset(train_full,train_transform,'imagenet'),batch_size=BATCH_SIZE,
                      shuffle=True,drop_last=False,num_workers=0,
                      generator=torch.Generator().manual_seed(SEED))
    logs=[]
    for epoch in range(1,max(epochs.values())+1):
        tally={r:[0.,0] for r in heads if epoch<=epochs[r]}
        for bi,(images,crop,disease,_) in enumerate(loader,1):
            with torch.no_grad():feats=backbone(images.to(DEVICE)).float()
            crop,disease=crop.to(DEVICE),disease.to(DEVICE)
            for name in tally:
                loss,_,_=train_one_step(heads[name],optimizers[name],feats,crop,disease,prototype,name)
                tally[name][0]+=loss*len(images);tally[name][1]+=len(images)
            if bi==1 or bi%75==0 or bi==len(loader):
                print(f'refit epoch {epoch}/{max(epochs.values())}, batch {bi}/{len(loader)}',flush=True)
        for name,values in tally.items():
            schedulers[name].step()
            if values[1]!=len(train_full):
                raise RuntimeError('Full refit did not train on all training rows.')
            logs.append({'recipe':name,'epoch':epoch,'loss':values[0]/values[1], 'images':values[1]})
    FINAL_DIR.mkdir(parents=True,exist_ok=True)
    CKPT_DIR.mkdir(parents=True,exist_ok=True)
    torch_atomic(BACKBONE_PATH,backbone.state_dict())
    records={}
    for name,head in heads.items():
        path=CKPT_DIR/f'{name}.pt'
        torch_atomic(path,head.cpu().state_dict())
        records[name]={'epoch':epochs[name],'head_filename':path.name,
                       'head_sha256':sha256_file(path)}
    pd.DataFrame(logs).to_csv(FINAL_DIR/'refit_history.csv',index=False)
    lock={'development_sha256':sha256_file(DEV_DIR/'development_selection.json'),
          'settings':SETTINGS, 'primary_recipe':development['primary_recipe'],
          'gamma':development['gamma'], 'heads':records,
          'backbone_sha256':sha256_file(BACKBONE_PATH),
          'seen_pairs':seen_pairs.tolist(), 'full_train_count':len(train_full)}
    json_atomic(LOCK_PATH,lock)
    print('Full-train model locked:',LOCK_PATH)
    return lock

lock = full_training_refit(development)

## 6 · Optional official evaluation (manual trigger only)

**Do not enable just to compare multiple experimental versions.** Set `RUN_FINAL_EVALUATION=True` only after the recipe, epoch and γ are locked and your lecturer has confirmed the evaluation plan. The output reports B0 and the selected proposed recipe under **raw and calibrated** scoring. Prediction before labels is enforced by the code sequence. The 333 candidates include the 10 unseen `(crop,disease)` identities in the final stage only.

In [ ]:
def final_candidate_pairs():
    # Allowed only in official final evaluation. No pair_label column is read.
    frame=pd.read_csv(team.TEST_UNSEEN_CSV,header=None,usecols=[1,2],dtype=str,
                      keep_default_na=False)
    frame.columns=['crop','disease']
    if not frame.empty and not frame.iloc[0]['crop'].strip().isdigit():frame=frame.iloc[1:]
    frame=frame.astype('int64').drop_duplicates().sort_values(['crop','disease'])
    extra=torch.as_tensor(frame[['crop','disease']].to_numpy(copy=True),dtype=torch.long)
    pairs=torch.cat([seen_pairs,extra])
    if len(extra)!=10 or len(pairs.unique(dim=0))!=333:
        raise ValueError('Final candidate set must contain exactly 323+10 unique pairs.')
    if not set(extra[:,0].tolist())<=set(seen_pairs[:,0].tolist()):
        raise ValueError('Official unseen crop outside training concepts.')
    if not set(extra[:,1].tolist())<=set(seen_pairs[:,1].tolist()):
        raise ValueError('Official unseen disease outside training concepts.')
    return pairs

def final_filenames(path):
    frame=pd.read_csv(path,header=None,usecols=[0],dtype=str,keep_default_na=False)
    vals=frame.iloc[:,0].tolist()
    headers={'image','filename','file_name','path','image_path','image_name','file_path'}
    if vals and (vals[0].strip().lower() in headers or
                 (len(vals)>1 and not Path(vals[0]).suffix and Path(vals[1]).suffix)):
        vals=vals[1:]
    names=[image_key(x) for x in vals]
    if not names or len(set(names))!=len(names):raise ValueError('Invalid official image names.')
    return names

class OfficialImages(Dataset):
    def __init__(self,names):self.names=names
    def __len__(self):return len(self.names)
    def __getitem__(self,index):
        with Image.open(team.IMAGE_DIR/self.names[index]) as im:
            return eval_transform(im.convert('RGB'))

@torch.inference_mode()
def infer_official(backbone,heads,candidates,names,perturb=None):
    dl=DataLoader(OfficialImages(names),batch_size=BATCH_SIZE,shuffle=False,num_workers=0)
    means,stds=(torch.tensor(v).view(1,3,1,1) for v in team.NORM_STATS['imagenet'])
    result={r:[] for r in heads}
    for images in dl:
        if perturb is not None:
            pixels=(images*stds+means).clamp(0,1)
            images=(perturb(pixels).clamp(0,1)-means)/stds
        feats=backbone(images.to(DEVICE))
        for r,h in heads.items():
            crop,disease,_,_=h(feats)
            values=compose_scores(crop,disease,candidates).cpu()
            if values.shape!=(len(images),333) or not torch.isfinite(values).all():
                raise ValueError('Official prediction must be finite [N,333].')
            result[r].append(values)
    return {name:torch.cat(parts) for name,parts in result.items()}

def ground_truth_after_predictions(path,names,map_pairs,seen):
    # This is ONLY called after score arrays and top-k predictions are completed.
    frame=pd.read_csv(path,header=None,usecols=[0,1,2],dtype=str,keep_default_na=False)
    frame.columns=['image','crop','disease']
    if not frame.empty and not frame.iloc[0]['crop'].strip().isdigit():frame=frame.iloc[1:]
    if frame.image.map(image_key).tolist()!=names:
        raise ValueError('Prediction image ordering differs from ground truth order.')
    labels=torch.tensor([map_pairs[(int(c),int(d))] for c,d in
                         frame[['crop','disease']].itertuples(index=False,name=None)])
    if not ((labels<323)==seen).all():raise ValueError('Incorrect seen/unseen split labels.')
    return labels

def score_topk(predictions,labels,k):
    return round(100*predictions[k].eq(labels[:,None]).any(1).float().mean().item(),2)

def official_evaluation():
    if DEBUG or not RUN_FINAL_EVALUATION:
        print('Official evaluation disabled. Set DEBUG=False and RUN_FINAL_EVALUATION=True only for final testing.')
        return None
    if not LOCK_PATH.is_file():raise FileNotFoundError('Final model lock missing.')
    locked=json.loads(LOCK_PATH.read_text(encoding='utf-8'))
    if locked['settings']!=SETTINGS or locked['seen_pairs']!=seen_pairs.tolist():
        raise RuntimeError('Model settings/candidate identities changed after locking.')
    if locked['development_sha256']!=sha256_file(DEV_DIR/'development_selection.json'):
        raise RuntimeError('Development selection changed since lock.')
    if locked['backbone_sha256']!=sha256_file(BACKBONE_PATH):
        raise RuntimeError('Frozen backbone checkpoint changed.')
    for r,rec in locked['heads'].items():
        if rec['head_sha256']!=sha256_file(CKPT_DIR/rec['head_filename']):
            raise RuntimeError('Head checkpoint changed.')
    report_path=OFFICIAL_DIR/'official_comparison.csv'
    if report_path.exists():raise FileExistsError('Official evaluation was already run for this lock.')
    # All model selection is finished. Construct candidate identities for final evaluation.
    candidates=final_candidate_pairs()
    names_s=final_filenames(team.TEST_SEEN_CSV)
    names_u=final_filenames(team.TEST_UNSEEN_CSV)
    if set(names_s)&set(names_u) or set(train_full.image)&(set(names_s)|set(names_u)):
        raise ValueError('Train/test filename overlap.')
    backbone=timm.create_model(BACKBONE,pretrained=False,num_classes=0,
                               img_size=team.IMAGE_SIZE).to(DEVICE)
    backbone.load_state_dict(torch.load(BACKBONE_PATH,map_location='cpu',weights_only=True),strict=True)
    backbone.eval().requires_grad_(False)
    heads={}
    for recipe,rec in locked['heads'].items():
        head=Heads(backbone.num_features,RECIPES[recipe][0]).to(DEVICE)
        head.load_state_dict(torch.load(CKPT_DIR/rec['head_filename'],map_location='cpu',weights_only=True))
        head.eval().requires_grad_(False)
        heads[recipe]=head
    with torch.no_grad():
        raw_s=infer_official(backbone,heads,candidates,names_s)
        raw_u=infer_official(backbone,heads,candidates,names_u)
    membership=(torch.arange(333)<323).float()
    predicted={}
    for recipe in heads:
        for scoring,gamma in [('raw',0.0),('calibrated',locked['gamma'][recipe])]:
            seen_score=raw_s[recipe]-gamma*membership
            unseen_score=raw_u[recipe]-gamma*membership
            predicted[(recipe,scoring)]={
                'seen':{k:seen_score.topk(k,dim=1).indices for k in (1,5)},
                'unseen':{k:unseen_score.topk(k,dim=1).indices for k in (1,5)},
            }
    # IMPORTANT: all raw + calibrated official predictions have been generated.
    pair_to_row={tuple(pair):i for i,pair in enumerate(candidates.tolist())}
    y_s=ground_truth_after_predictions(team.TEST_SEEN_CSV,names_s,pair_to_row,True)
    y_u=ground_truth_after_predictions(team.TEST_UNSEEN_CSV,names_u,pair_to_row,False)
    y_all=torch.cat([y_s,y_u])
    records=[]
    for (recipe,scoring),pred in predicted.items():
        row={'recipe':recipe,'scoring':scoring,'gamma':0.0 if scoring=='raw' else locked['gamma'][recipe],
             'epoch':locked['heads'][recipe]['epoch'],'candidates':333}
        for k in (1,5):
            row[f'seen_top{k}']=score_topk(pred['seen'],y_s,k)
            row[f'unseen_top{k}']=score_topk(pred['unseen'],y_u,k)
            both=torch.cat([pred['seen'][k],pred['unseen'][k]])
            row[f'overall_top{k}']=round(100*both.eq(y_all[:,None]).any(1).float().mean().item(),2)
        row['harmonic_top1']=harmonic_mean(row['seen_top1'],row['unseen_top1'])
        row['relative_unseen_gain_vs_historical_B0_pct']=round(100*(row['unseen_top1']/9.6-1),2)
        row['target_20pct_met']=row['unseen_top1']>=20
        row['target_30pct_met']=row['unseen_top1']>=30
        records.append(row)
    OFFICIAL_DIR.mkdir(parents=True,exist_ok=True)
    pd.DataFrame(records).to_csv(report_path,index=False)
    json_atomic(OFFICIAL_DIR/'official_report.json',{
        'checkpoint_lock_sha256':sha256_file(LOCK_PATH),
        'protocol':'all 333 candidates for every image; truth loaded only after predictions',
        'results':records,'counts':{'seen':len(y_s),'unseen':len(y_u)}})
    # Team confusion writer expects its canonical candidate ordering: map only after predictions.
    canonical=get_pairs(final_evaluation=True)
    canonical_rows={tuple(v):i for i,v in enumerate(canonical.tolist())}
    if set(canonical_rows)!=set(pair_to_row):raise ValueError('Canonical candidate mismatch.')
    translate=torch.tensor([canonical_rows[tuple(x)] for x in candidates.tolist()])
    for (recipe,scoring),pred in predicted.items():
        s_winners=translate[pred['seen'][1].squeeze(1)]
        s_targets=translate[y_s]
        votes=F.one_hot(s_winners,num_classes=333).float()
        stem=(OFFICIAL_DIR/f'{recipe}_{scoring}').relative_to(team.RESULTS_DIR).as_posix()
        save_confusion(votes,s_targets,stem,final_evaluation=True)
    return pd.DataFrame(records)

if not RUN_FINAL_EVALUATION:
    print('Official evaluation OFF. Train-only development and final refit are complete (if DEBUG=False).')
else:
    display(official_evaluation())

### Manual final evaluation, without retraining

After completing `DEBUG=False` training **in the same Jupyter kernel** and receiving permission to perform the final test, run **only the cell below**. Do **not** rerun the Setup/Development/Refit cells; they create a new run ID. This cell is disabled by default. The official evaluation is for a locked method, not for iterative tuning.

In [ ]:
# Only after your final methodology has been approved and frozen:
# RUN_FINAL_EVALUATION = True
# display(official_evaluation())

## 7 · Reporting checklist

1. Show the **matched B0** and **chosen proposed model** (raw and calibrated as different rows); use **323 seen + 10 unseen = 333** for *every* final prediction.
2. State clearly which changes are architecture/features (`E1`), cross-crop prototype regularisation (`E2`), cross-crop SupCon (`E3`) and prediction-only calibration (γ).
3. The 20–30% unseen Top-1 is a **target**, not a demonstrated result. Verify with an untouched test or disclose previous official test exposure.
4. Use the **independent training-only proxy check** to judge real progress before testing; avoid selecting from official test scores.
5. B0 is matched to Baseline 2 architecture and settings but its exact result may not reproduce the published reference due to implementation/version differences; report your actually rerun B0 and release value separately.
6. The prototype loss only applies when a disease occurs in **more than one training crop**. Its coverage is logged; if it helps less than expected, report the restriction rather than tune on official unseen labels.

**Files:** `results/approach2_a6_disentangled_proto/<run_id>/development/`, `final/`, `official/`; checkpoints saved separately under `checkpoints/approach2_a6_disentangled_proto/<run_id>/`.